In [1]:
import pprint

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware
agent=create_agent(
    model="groq:llama-3.3-70b-versatile",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:llama-3.3-70b-versatile",
            trigger=("tokens",100),
            keep=("messages",1),
        ),

    ]

)

In [4]:
from langchain.messages import HumanMessage,AIMessage
response=agent.invoke(
    {"messages":[
        HumanMessage(content="what is the captial of moon?"),
        AIMessage(content="The captal of moon is Lunapolis"),
        HumanMessage(content="what is the wheather of Lunapolis"),
        AIMessage(content="skies are clear ,with a high of 120c and low of -100c"),
        HumanMessage(content="How many cheses miners live in Lunapolis?"),
        AIMessage(content="They are 1000000 cheese miners are living in Lunapolis"),
        HumanMessage(content="Do you think the cheese miners Union will strike?"),
        AIMessage(content="yes they have un happy with there president"),
        HumanMessage(content="If you have lunapolis new presedint how would you respond to the cheese miners,Union?")
    ]
     },
    {"configurable":{"thread_id":"1"}}
)
print(response['messages'][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user's primary goal is to gather information about Lunapolis, a supposed capital of the Moon, including its weather, population of cheese miners, and potential labor issues, as well as to explore scenarios related to the cheese miners' union and the city's leadership.

## SUMMARY
The conversation history reveals the following key points: 
- The capital of the Moon is claimed to be Lunapolis.
- The weather in Lunapolis is described as having clear skies with extreme temperatures, ranging from 120°C high to -100°C low.
- There are allegedly 1,000,000 cheese miners living in Lunapolis.
- The cheese miners are unhappy with their president, suggesting a potential for the cheese miners' union to strike.
- A hypothetical scenario was explored where a new president of Lunapolis addresses the concerns of the cheese miners' union, proposing initiatives such as temperature regulation, fair compensation, safety and health measur

In [8]:
# ============================================================
# CELL 5: Define a @before_agent Middleware to Delete ToolMessages
# ============================================================
# @before_agent marks this function as pre-turn middleware.
#
# The function signature:
#   state: AgentState  → the full current state (messages, custom fields, etc.)
#   runtime: Runtime   → access to context, config, and other runtime data
#   returns: dict | None
#     → Return a dict of state updates to apply before the agent runs.
#     → Return None to make no changes.
#
# RemoveMessage(id=m.id):
#   → A special message type that tells LangGraph to DELETE the message
#     with that ID from the state.
#   → It does NOT add a message — it removes one.
#   → The ID (m.id) is the unique identifier assigned when the message
#     was created (UUID generated automatically by LangChain).
#
# Pattern used here:
#   1. Find all ToolMessages in the current state
#   2. Create a RemoveMessage for each one
#   3. Return them in the messages update list
#   4. LangGraph deletes those messages before the next LLM call
#
# This is useful when historical tool call results are bulky
# (e.g. raw Tavily JSON or large SQL result sets) and no longer
# needed for the current turn's reasoning.

from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage
from pprint import pprint

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]
    # Find all ToolMessages (raw tool call results) in history
    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]

    if tool_messages:
        # Return RemoveMessage for each — LangGraph deletes them from state
        return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}
    else:
        return {"messages": messages}

In [6]:
agent=create_agent(
    model="groq:llama-3.3-70b-versatile",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages]
)

In [9]:
response=agent.invoke(
    {"messages":[
        HumanMessage(content="my device want turn on what should I Do?"),
        ToolMessage(content="blorp-x7 intiating diagonstic ping",tool_call_id="2"),
        AIMessage(content="Is this device plugged in and turned on"),
        ToolMessage(content="temp-42c voltage=2.9v-greedible complete",tool_call_id="2"),
        HumanMessage(content="what is the temprature of device?")

    ]},
    {"configurable":{"thread_id":"2"}}
)
pprint(response)

{'messages': [HumanMessage(content='my device want turn on what should I Do?', additional_kwargs={}, response_metadata={}, id='5c998ca0-b89c-41d3-b4a6-0be7efe9a6dd'),
              AIMessage(content='Is this device plugged in and turned on', additional_kwargs={}, response_metadata={}, id='1642e90c-baef-44b5-b91b-ae01b4ec2556', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content='what is the temprature of device?', additional_kwargs={}, response_metadata={}, id='cb79bbaf-e9a8-4fe1-99be-fe379745392a'),
              AIMessage(content="If your device won't turn on, it's possible that it may have overheated. However, without knowing the specific device, it's hard to provide an exact temperature range.\n\n Generally, most devices can operate safely within the following temperature ranges:\n\n* Smartphones: 32°F to 95°F (0°C to 35°C)\n* Laptops: 50°F to 95°F (10°C to 35°C)\n* Tablets: 32°F to 95°F (0°C to 35°C)\n\nIf you suspect that your device has overheated, try the 

In [10]:
print(response['messages'][-1].content)

If your device won't turn on, it's possible that it may have overheated. However, without knowing the specific device, it's hard to provide an exact temperature range.

 Generally, most devices can operate safely within the following temperature ranges:

* Smartphones: 32°F to 95°F (0°C to 35°C)
* Laptops: 50°F to 95°F (10°C to 35°C)
* Tablets: 32°F to 95°F (0°C to 35°C)

If you suspect that your device has overheated, try the following:

1. Let it cool down for at least 30 minutes.
2. Remove any cases or covers that may be trapping heat.
3. Avoid exposing your device to direct sunlight or high-temperature environments.

If your device still won't turn on after cooling down, try the following troubleshooting steps:

1. Charge your device for at least 30 minutes.
2. Check the power button for any debris or damage.
3. Try booting your device in safe mode (if applicable).
4. If none of the above steps work, consider visiting an authorized service center or contacting the manufacturer's su